# sPTC on a real GPU: Kaggle T4 validation

Checks the simulator's serving claims on real vLLM. It runs **Qwen3-1.7B (fp16) on one Kaggle T4**, and takes about 1.5 hours.

**Settings:** Accelerator **GPU T4** (one is enough), Internet **on**.

**Input:** add the dataset `sptc-bundle` (made with `kaggle/make_bundle.sh`). If the code is
on GitHub instead, set `REPO` in the first cell.

**What it does:**
1. **Calibrate** on solo requests: batch-1 decode time and prefill throughput.
2. **Predict**: the simulator writes its predictions, timestamped, *before* step 3.
3. **Measure concurrency**: the decode free window (is batch 2–8 nearly free on CUDA?), prefill interference, fan-out makespan.
4. **spec-ptc suite** with per-call token logging (the data `runs.csv` lacks).
5. **Load test**: root model served by the same vLLM, spec-ptc vs no speculation, with
   and without one bad guess per speculation.
6. **Scheduler plugin**: the same load test with the speculation-aware scheduler, plus
   its vLLM unit tests.

Download `/kaggle/working/sptc_out.zip` at the end.


In [ ]:
import os, subprocess, sys, time, json, glob, shutil, zipfile

REPO = ""   # optional GitHub URL; leave empty to use the uploaded dataset

def find_bundle(top="/kaggle/input"):
    # a folder that contains both sptc-sim and spec-ptc, at any depth
    for root, dirs, files in os.walk(top):
        if "sptc-sim" in dirs and "spec-ptc" in dirs:
            return root
    # zip uploaded but not unpacked: unpack it into /kaggle/working and search again
    for z in glob.glob(f"{top}/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as f:
            f.extractall("/kaggle/working/_unzipped")
        return find_bundle("/kaggle/working/_unzipped")
    return None

if REPO:
    B = "/kaggle/working/sptc-bundle"
    subprocess.run(["git", "clone", "--depth", "1", REPO, f"{B}/sptc-sim"], check=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/alexzhang13/spec-ptc",
                    f"{B}/spec-ptc"], check=True)
else:
    src = find_bundle()
    if src is None:
        print("Bundle not found. Here is what is under /kaggle/input:")
        for root, dirs, files in os.walk("/kaggle/input"):
            if root.count(os.sep) <= 6:
                print(root, dirs[:8], files[:5])
        raise SystemExit("bundle not found: see the listing above")
    print("found bundle at", src)
    B = "/kaggle/working/sptc-bundle"
    shutil.copytree(src, B, dirs_exist_ok=True)  # inputs are read-only

SIM, SPTC = f"{B}/sptc-sim", f"{B}/spec-ptc"
OUT = "/kaggle/working/sptc_out"
os.makedirs(OUT, exist_ok=True)
print(SIM, os.listdir(SIM))
print(SPTC, os.listdir(SPTC))
!nvidia-smi

In [ ]:
%%bash -s "$SPTC"
pip -q install "vllm==0.32.0" 2>&1 | tail -2
pip -q install pandas numpy pytest 2>&1 | tail -1
pip -q install -e "$1" 2>&1 | tail -1   # spec-ptc (the sptc-hints version if bundled)
# Kaggle preinstalls torchaudio built for an older CUDA; vLLM's torch upgrade makes it fail
# at import (transformers imports it), so remove it: nothing here uses audio
pip -q uninstall -y torchaudio 2>&1 | tail -1
python -c "import torchvision.ops, torchvision; print('torchvision', torchvision.__version__, 'OK')"
python -c "import vllm, torch, spec_ptc; print('vllm', vllm.__version__, 'torch', torch.__version__, torch.cuda.get_device_name(0))"

In [ ]:
import subprocess, time, urllib.request
MODEL = "Qwen/Qwen3-1.7B"
def start_vllm(extra=(), log="vllm.log", env=None):
    cmd = ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", "8192",
           "--gpu-memory-utilization", "0.90", "--port", "8000", *extra]
    p = subprocess.Popen(cmd, stdout=open(f"{OUT}/{log}", "w"), stderr=subprocess.STDOUT,
                         env={**os.environ, **(env or {})})
    for _ in range(600):
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2); print("up"); return p
        except Exception:
            if p.poll() is not None: raise RuntimeError(open(f"{OUT}/{log}").read()[-3000:])
            time.sleep(2)
    raise TimeoutError
def stop_vllm(p):
    p.terminate(); p.wait(timeout=120); time.sleep(10)
srv = start_vllm()

In [ ]:
BENCH = f"python {SIM}/kaggle/vllm_bench.py"
!{BENCH} calib   --model {MODEL} --out {OUT}
!{BENCH} predict --out {OUT} --gpu T4
!cat {OUT}/predict.json

In [ ]:
# measured AFTER predict.json was written
!{BENCH} concur --model {MODEL} --out {OUT}

In [ ]:
# spec-ptc suite with per-call token logging (3 reps; ~15 min)
!{BENCH} suite --model {MODEL} --out {OUT} --spec-ptc {SPTC} --reps 3

In [ ]:
# load test: root served by vLLM; baseline vs spec-ptc, 0 and 1 bad guess per speculation (~30 min)
!{BENCH} load --model {MODEL} --out {OUT} --spec-ptc {SPTC} --agents 1,4,8 --decoys 0,1 --turns 4 \
    --patterns map16,map_reduce,two_stage,side_branch_chain
stop_vllm(srv)

## The speculation-aware scheduler plugin

1. Run the plugin's tests: its local policy tests, plus the vLLM-dependent scheduler tests.
   These include a step-by-step differential test against stock vLLM on untagged traffic.
2. Run the end-to-end smoke test (S1–S4) through the OpenAI API.
3. Repeat the load test with requests tagged: decoys as `bet`, spec-ptc dispatches as
   `early`.

Flags are from `integrations/README.md`.

In [ ]:
PLUGIN_ENV = {"PYTHONPATH": f"{SIM}/integrations:" + os.environ.get("PYTHONPATH", "")}
!pip -q install -e {SIM}/integrations --no-deps
!cd {SIM}/integrations && python -m pytest -q -p no:cacheprovider tests tests_vllm 2>&1 | tail -25

In [ ]:
SMOKE = f"python {SIM}/integrations/kaggle/sptc_vllm_smoke.py --model {MODEL}"
base_flags = ["--scheduler-cls", "sptc_vllm.scheduler.SptcScheduler", "--enforce-eager"]
srv = start_vllm(base_flags + ["--additional-config", json.dumps({"sptc": {"free_window": 2}})],
                 log="vllm_smoke123.log", env=PLUGIN_ENV)
cmd, log = f"{SMOKE} --cases s1,s2,s3", f"{OUT}/smoke_s123.jsonl"
!{cmd} 2>&1 | grep check | tee {log}
stop_vllm(srv)
# S4 needs one request of ~600 prompt + 1500 output tokens to fit (max-model-len 2560) in a
# small KV cache (200 blocks x 16 = 3200 tokens; exactly 2560 fails vLLM's startup check),
# so the 4 requests together (~7900 tokens) overflow it and a running bet must be dropped
srv = start_vllm(base_flags + ["--num-gpu-blocks-override", "200", "--max-model-len", "2560",
                               "--additional-config", json.dumps({"sptc": {}})],
                 log="vllm_smoke4.log", env=PLUGIN_ENV)
cmd, log = f"{SMOKE} --cases s4", f"{OUT}/smoke_s4.jsonl"
!{cmd} 2>&1 | grep check | tee {log}
stop_vllm(srv)
!grep -c Traceback {OUT}/vllm_smoke123.log {OUT}/vllm_smoke4.log || true

In [ ]:
# tagged load test (same settings as the untagged one above)
PLUGIN_FLAGS = ["--scheduler-cls", "sptc_vllm.scheduler.SptcScheduler",
                "--additional-config", json.dumps({"sptc": {"free_window": 8, "kv_reserve": 0.10,
                                                            "spec_prefill_budget": 512}})]
srv = start_vllm(PLUGIN_FLAGS, log="vllm_plugin.log", env=PLUGIN_ENV)
!{BENCH} load --model {MODEL} --out {OUT} --spec-ptc {SPTC} --agents 1,4,8 --decoys 0,1 --turns 4 \
    --patterns map16,map_reduce,two_stage,side_branch_chain --tag-bets
stop_vllm(srv)

In [ ]:
!cd /kaggle/working && zip -qr sptc_out.zip sptc_out && ls -la sptc_out.zip
print(open(f"{OUT}/predict.json").read()[:1500])